# Chat with an LLM: a conversation with a graph

Tutorial 03 asked a graph one question at a time. Every question stood on its own, because
`Cycle.RDFQueryByPrompt` takes one prompt and remembers nothing.

This one holds a **conversation**: you can ask *"and on the ground floor?"* and be understood,
and you can ask for a change and be shown it before it lands. The graph is edited only when you
say so, and written only to a copy.

The interesting part is *where* the memory lives. It is not in the query pipeline — that stays
stateless, and an answer is still written from retrieved rows and nothing else. The memory sits
in one step in front of it, and this notebook is mostly about watching that step work.

**Prerequisites**

```bash
pip install "btwin[llm,rdf]"
export OPENROUTER_API_KEY=sk-or-...
```

**Two things to expect**

- **This costs money.** A full run is a fraction of a cent on the default model. `CostMeter`
  tracks it and the last section prints the total.
- **The outputs below will not reproduce exactly.** The model is free to word an answer
  differently on your run. Where it matters, this notebook *checks* its work.

In [1]:
from pathlib import Path

import btwin
from btwin import (LLM, RDF, Cycle, CostMeter, NetworkX, SpatialElement, Tool)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

BASE = "https://example.org/riverside/"

llm = LLM.Constructor()
meter = CostMeter()

print("BTwin", btwin.__version__)
print("model:", llm.model_name)

C:\Users\massa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Couldn't import dot_parser, loading of dot files will not be possible.


BTwin 0.5.6
model: google/gemini-2.5-flash-lite


## 1. A graph to talk to

No model is involved here. This is the ordinary BTwin API from tutorial 01: a two-storey block,
four spaces, and a zone that groups the two upstairs rooms.

In [2]:
building = SpatialElement.Constructor("bld-01", "bot:Building", "Riverside Block")
ground = SpatialElement.Constructor("storey-00", "bot:Storey", "Ground Floor")
first = SpatialElement.Constructor("storey-01", "bot:Storey", "First Floor")

spaces = [
    SpatialElement.Constructor("space-01", "bot:Space", "Reception"),
    SpatialElement.Constructor("space-02", "bot:Space", "Canteen"),
    SpatialElement.Constructor("space-03", "bot:Space", "Open Office"),
    SpatialElement.Constructor("space-04", "bot:Space", "Meeting Room"),
]
zone = SpatialElement.Constructor("zone-01", "brick:Zone", "North Wing")

SpatialElement.SetLocationRelationship(ground, linkedObject=building)
SpatialElement.SetLocationRelationship(first, linkedObject=building)
for space, storey in zip(spaces, [ground, ground, first, first]):
    SpatialElement.SetLocationRelationship(space, linkedObject=storey)

# The two upstairs rooms are also in a zone, which gives the graph a second route upwards
SpatialElement.SetLocationRelationship(zone, linkedObject=first)
for space in spaces[2:]:
    SpatialElement.SetLocationRelationship(space, linkedObject=zone)

objects = [building, ground, first, *spaces, zone]

nxGraph = NetworkX.Constructor("MultiDiGraph", name="Riverside Block")
for obj in objects:
    NetworkX.AddNodeByObject(nxGraph, obj)
for obj in objects:
    NetworkX.AddEdgesByObject(nxGraph, obj)

graph, turtle = NetworkX.ToRDF(nxGraph, savePath=OUTPUT / "riverside.ttl", baseIRI=BASE)
print(f"{len(graph)} triples, {nxGraph.number_of_nodes()} nodes")

25 triples, 8 nodes


## 2. What the model is given

Two blocks of grounding, both computed without a model.

`RDF.SchemaSummary` describes the graph one hop at a time — its prefixes, classes, predicates,
the SHAPES list and the labelled entities.

In [3]:
schema = RDF.SchemaSummary(graph)

print(schema["text"][schema["text"].index("SHAPES"):schema["text"].index("ENTITIES")])

SHAPES (subject class -predicate-> object class)
  bot:Building -rdfs:label-> literal
  bot:Space -brick:hasLocation-> bot:Storey
  bot:Space -brick:hasLocation-> brick:Zone
  bot:Space -rdfs:label-> literal
  bot:Storey -brick:hasLocation-> bot:Building
  bot:Storey -rdfs:label-> literal
  brick:Zone -brick:hasLocation-> bot:Storey
  brick:Zone -rdfs:label-> literal




Read that list carefully. It says a `bot:Space` is located in a `bot:Storey`, and separately
that a `bot:Space` is located in a `brick:Zone`, and that a `brick:Zone` is located in a
`bot:Storey`. What it does **not** say is that those hops compose into two different routes
from a space to its storey.

Composing them is left to the model, and composing is where a generated query goes wrong.
`RDF.Chains` does the composing in advance:

In [4]:
chains = RDF.Chains(graph)

for chain in chains:
    print(f"[{chain['hops']} hops] {chain['template']}")
    print(f"          e.g. {chain['example']}")

[2 hops] bot:Space -brick:hasLocation-> bot:Storey -brick:hasLocation-> bot:Building
          e.g. 'Reception' -> 'Ground Floor' -> 'Riverside Block'
[2 hops] brick:Zone -brick:hasLocation-> bot:Storey -brick:hasLocation-> bot:Building
          e.g. 'North Wing' -> 'First Floor' -> 'Riverside Block'
[3 hops] bot:Space -brick:hasLocation-> brick:Zone -brick:hasLocation-> bot:Storey -brick:hasLocation-> bot:Building
          e.g. 'Open Office' -> 'North Wing' -> 'First Floor' -> 'Riverside Block'


Every one of those was found by walking real triples, so a route the data does not actually take
cannot appear — and the example is what fixes the direction. *"Open Office is located in North
Wing"* cannot be read backwards, while an arrow between two class names can.

This is what `Cycle.RDFQueryByPrompt` adds to the schema before asking for a query. It costs
tokens, so it is worth knowing how many:

In [5]:
block = Tool.RDFChainBlock(chains)

print(f"schema : {len(schema['text']):>5} characters")
print(f"chains : {len(block):>5} characters")
print(f"total  : {len(schema['text']) + len(block):>5} characters of grounding per query")

schema :  1880 characters
chains :   833 characters
total  :  2713 characters of grounding per query


## 3. The step that remembers: the router

Every chat turn goes through `Tool.ChatRoute` first. It reads the conversation so far and the
new message, and returns two things: what kind of turn this is, and the message **restated so it
stands on its own**.

Start with no history at all, and an unambiguous question:

In [6]:
first_turn = Tool.ChatRoute(llm, Tool.ChatTranscript([]), "Which spaces are on the first floor?",
                            meter)
print(first_turn)

{'intent': 'question', 'request': 'Which spaces are on the first floor?'}


Nothing to resolve, so the restatement is the question. Now the interesting case — a follow-up
that means nothing on its own:

In [7]:
history = [{
    "message": "Which spaces are on the first floor?",
    "intent": "question",
    "request": "Which spaces are on the first floor?",
    "answer": "Open Office and Meeting Room are on the First Floor.",
}]

follow_up = Tool.ChatRoute(llm, Tool.ChatTranscript(history), "and the ground floor?", meter)
print(follow_up)

{'intent': 'question', 'request': 'Which spaces are on the ground floor?'}


`and the ground floor?` became a whole question. **That restatement is what reaches
`RDFQueryByPrompt`** — which is why the query pipeline needs no memory of its own, and why the
transcript never enters the window where an answer is written from retrieved rows.

The transcript it read is plain text, built with no model call:

In [8]:
print(Tool.ChatTranscript(history))

[1] user: Which spaces are on the first floor?
     assistant: Open Office and Meeting Room are on the First Floor.


Only the last `maxTurns` turns survive and each answer is cut short, so a long conversation does
not bill the whole history back to the first message on every turn.

## 4. One turn at a time

`Cycle.RDFChatTurn` puts that together: route the message, then send the restated request to
`RDFQueryByPrompt` to be answered or to `RDFEditByPrompt` to be applied. It reads no keyboard and
prints nothing, and it does not modify the history you give it — the new one comes back in the
result.

In [9]:
history, schema, chains = [], schema, chains
turn = Cycle.RDFChatTurn(graph, "Which spaces are on the first floor?",
                         history=history, llm=llm, schema=schema, chains=chains, meter=meter)
history, schema, chains = turn["history"], turn["schema"], turn["chains"]

print("intent :", turn["intent"])
print("answer :", turn["answer"])
print()
print(turn["sparql"])

intent : question
answer : The spaces on the first floor are space-03 and space-04.

PREFIX bot: <https://w3id.org/bot#>
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?space
WHERE {
  ?space rdf:type bot:Space .
  ?space brick:hasLocation <https://example.org/riverside/storey-01> .
}
LIMIT 100


Read the query, not just the sentence.

On the run committed here the model wrote `SELECT ?space` and never bound the label, so the
answer names `space-03` and `space-04` instead of *Open Office* and *Meeting Room*. The query is
correct — those really are the two spaces on the first floor — but the projection is not what a
person asked for.

Nothing in the pipeline can catch this. It parses, its vocabulary is legal, it returns rows, and
the answer agent faithfully reports the rows it was given. This is why the chat prints the query
under every answer by default, and why `RDFChatTurn` returns it in `["sparql"]`.

Now the follow-up. Watch `request` rather than the answer — that is the memory doing its work:

In [10]:
turn = Cycle.RDFChatTurn(graph, "and the ground floor?",
                         history=history, llm=llm, schema=schema, chains=chains, meter=meter)
history, schema, chains = turn["history"], turn["schema"], turn["chains"]

print("you typed  :", turn["history"][-1]["message"])
print("understood :", turn["request"])
print("answer     :", turn["answer"])

you typed  : and the ground floor?
understood : Which spaces are on the ground floor?
answer     : The Reception and Canteen are on the ground floor.


Check it rather than believing it. Here is the same question written by hand:

In [11]:
truth = RDF.Query(graph, """
PREFIX bot:   <https://w3id.org/bot#>
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?spaceName WHERE {
  ?storey rdfs:label "Ground Floor" .
  ?space  brick:hasLocation ?storey ; rdfs:label ?spaceName .
}""")

print("actually on the ground floor:", sorted(row["spaceName"] for row in truth))
print("the model answered          :", turn["answer"])

actually on the ground floor: ['Canteen', 'Reception']
the model answered          : The Reception and Canteen are on the ground floor.


## 5. A turn that asks nothing of the graph

Not every message is a question about the building. A greeting, or a question about the
conversation itself, is routed to `talk` and answered from the transcript alone — no SPARQL, no
query, nothing retrieved.

This matters more than it looks. Without it, *"what did I just ask?"* becomes a query for a
building fact that does not exist, and comes back as a confident *"there are no results"*.

In [12]:
turn = Cycle.RDFChatTurn(graph, "What did I ask you first?",
                         history=history, llm=llm, schema=schema, chains=chains, meter=meter)
history = turn["history"]

print("intent :", turn["intent"])
print("sparql :", repr(turn["sparql"]))
print("answer :", turn["answer"])

intent : talk
sparql : ''
answer : You first asked which spaces are on the first floor. You also asked which spaces are on the ground floor.


## 6. Editing, with the change shown first

An edit turn goes to `Cycle.RDFEditByPrompt`, which writes a SPARQL `UPDATE` and runs it against
a **copy**. Your graph does not move until a `confirm` callable says so.

Pass no `confirm` at all and the edit is proposed but never applied — which is how you see what
an instruction *would* do:

In [13]:
before = len(graph)
proposed = Cycle.RDFChatTurn(graph, "Add a space called Archive on the ground floor.",
                             history=history, llm=llm, schema=schema, chains=chains, meter=meter)

print("intent  :", proposed["intent"])
print("applied :", proposed["applied"])
print("triples :", before, "->", len(graph), "(unchanged)")
print()
for triple in proposed["added"]:
    print("   +", " ".join(triple))

intent  : edit
applied : False
triples : 25 -> 25 (unchanged)

   + https://example.org/riverside/space-05 rdf:type bot:Space
   + https://example.org/riverside/space-05 rdfs:label Archive
   + https://example.org/riverside/space-05 brick:hasLocation https://example.org/riverside/storey-00


The `confirm` callable receives that same diff and returns a verdict. In the terminal chat it
prints the triples and reads `y/N`; here it can be anything — a rule, a prompt, or a flat yes:

In [14]:
def confirm(proposal):
    """Accept an edit only if it adds triples and removes none."""
    print(f"   proposal: +{len(proposal['added'])} / -{len(proposal['removed'])}")
    return bool(proposal["added"]) and not proposal["removed"]


turn = Cycle.RDFChatTurn(graph, "Add a space called Archive on the ground floor.",
                         history=history, llm=llm, schema=schema, chains=chains, meter=meter,
                         confirm=confirm)
history, schema, chains = turn["history"], turn["schema"], turn["chains"]

print("applied :", turn["applied"])
print("triples :", before, "->", len(graph))
print("answer  :", turn["answer"])

   proposal: +3 / -0
applied : True
triples : 25 -> 28
answer  : Done: 3 triple(s) added, 0 removed. The graph now holds 28 triples.


Two things happened there that are easy to miss.

The triples went into **your** graph object, not a replacement for it — the edit ran on a copy,
and the difference between the two was applied to the original, so anything else holding a
reference to `graph` sees the change too.

And `schema` and `chains` were rebuilt. Both describe the graph *as it was*, so a stale one would
ground the next question in a graph that has moved. Here is the proof:

In [15]:
print("Archive is in the refreshed schema:", "Archive" in schema["text"])

turn = Cycle.RDFChatTurn(graph, "Which spaces are on the ground floor now?",
                         history=history, llm=llm, schema=schema, chains=chains, meter=meter)
history = turn["history"]
print("answer:", turn["answer"])

Archive is in the refreshed schema: True


answer: The spaces on the ground floor are Reception, Canteen, and Archive.


## 7. The whole conversation, as the router sees it

Every turn so far, in the form the next router call would read:

In [16]:
print(Tool.ChatTranscript(history, maxTurns=20))

[1] user: Which spaces are on the first floor?
     assistant: The spaces on the first floor are space-03 and space-04.
[2] user: and the ground floor?
     understood as (question): Which spaces are on the ground floor?
     assistant: The Reception and Canteen are on the ground floor.
[3] user: What did I ask you first?
     assistant: You first asked which spaces are on the first floor. You also asked which spaces are on the ground floor.
[4] user: Add a space called Archive on the ground floor.
     assistant: Done: 3 triple(s) added, 0 removed. The graph now holds 28 triples.
[5] user: Which spaces are on the ground floor now?
     assistant: The spaces on the ground floor are Reception, Canteen, and Archive.


## 8. The same thing as a terminal chat

`Cycle.RDFChat` is the loop around `RDFChatTurn`, and the only part of `btwin.llm` that reads a
keyboard. It cannot run in a notebook — it blocks on input — so this section describes it.

```python
from btwin import RDF, Cycle

graph = RDF.ByTTL("riverside.ttl", baseIRI=BASE)
session = Cycle.RDFChat(graph, savePath="riverside_edited.ttl")
```

```text
Chatting about a graph of 23 triples, via google/gemini-2.5-flash-lite.
Ask a question, or describe a change. /help for commands, Esc or /exit to leave.

you> which spaces are on the first floor?

bot> Open Office and Meeting Room are on the First Floor.

     the query it ran:
       PREFIX bot: <https://w3id.org/bot#>
       ...
     this turn: 2 call(s), 1655+94 tokens, $0.000212
     so far:    2 call(s), 1655+94 tokens, $0.000212

you> add an archive next to the canteen

  the change it proposes:
    + https://example.org/riverside/space-05 rdf:type bot:Space
    + https://example.org/riverside/space-05 rdfs:label Archive
  apply? [y/N] y

bot> Done: 3 triple(s) added, 0 removed. The graph now holds 26 triples.
     saved to riverside_edited.ttl
```

The commands are `/sparql`, `/history`, `/schema`, `/cost`, `/silent`, `/verbose`, `/save` and
`/exit`. Escape leaves as well, and at the `apply?` prompt it means no — refusing a change is
not a reason to end the conversation.

**Where an edit is saved.** In memory as soon as you confirm it, and on disk as soon as it is
applied: `autoSave` is on by default, so a session cannot end with confirmed edits lost to a
forgotten `/save`. It writes to `savePath` and **never to the file the graph was read from** —
an edit is the model's work, and overwriting the source leaves nothing to compare it against.
A session that only asks questions writes nothing at all.

Two flags worth knowing: every turn prints the query it ran and what it cost, because the
reading that catches a wrong answer is the query rather than the sentence — `silent=True` turns
that off; `verbose=True` narrates each agent instead.

## 9. Saving this notebook's graph

The same write the chat performs, done by hand:

In [17]:
editedPath = OUTPUT / "riverside_edited.ttl"
graph.serialize(destination=str(editedPath), format="turtle")

original = RDF.ByTTL(OUTPUT / "riverside.ttl", baseIRI=BASE)
edited = RDF.ByTTL(editedPath, baseIRI=BASE)

print(f"riverside.ttl        : {len(original)} triples")
print(f"riverside_edited.ttl : {len(edited)} triples")
print(f"difference           : {len(edited) - len(original)} triples")

riverside.ttl        : 25 triples
riverside_edited.ttl : 28 triples
difference           : 3 triples


## 10. What it cost

In [18]:
total = meter.Total()

print(f"calls             : {total['calls']}")
print(f"prompt tokens     : {total['promptTokens']:,}")
print(f"completion tokens : {total['completionTokens']:,}")
print(f"cost              : {CostMeter.Format(total['cost'])}")
print()
for entry in meter.calls:
    print(f"   {entry['agent']:<16} {CostMeter.Describe(entry)}")

calls             : 17
prompt tokens     : 11,082
completion tokens : 883
cost              : $0.001461

   router           278+19 tokens, $0.000035
   router           298+19 tokens, $0.000037
   router           278+19 tokens, $0.000035
   agent 3 write    1036+111 tokens, $0.000148
   answer           107+17 tokens, $0.000017
   router           304+19 tokens, $0.000038
   agent 3 write    1036+151 tokens, $0.000164
   answer           116+11 tokens, $0.000016
   router           347+18 tokens, $0.000042
   talk             188+22 tokens, $0.000028
   router           389+21 tokens, $0.000047
   agent 1 edit     2342+125 tokens, $0.000284
   router           389+21 tokens, $0.000047
   agent 1 edit     2342+125 tokens, $0.000284
   router           431+20 tokens, $0.000051
   agent 3 write    1064+150 tokens, $0.000166
   answer           137+15 tokens, $0.000020


Notice how many calls the router accounts for: one per turn, on top of whatever the question or
the edit costs. That is the price of being understood when you say *"and the ground floor?"*.

## Questions to try

1. **Break the memory on purpose.** Ask a follow-up after ten unrelated turns, with
   `maxTurns=2`. The router no longer sees the turn the pronoun refers to — what does it
   restate the message as?
2. **Route something ambiguous.** "Can you add up the spaces on each floor?" contains the word
   *add*. Does it route to `question` or to `edit`? Read `turn["request"]`.
3. **Turn the chains off.** Re-run section 4 with `chains=[]` and compare the SPARQL. On a graph
   this small the model often needs no help; on one with two routes between the same classes it
   is a different story.
4. **Refuse an edit and ask about it.** Say no to a change, then ask "what did you want to
   change?" — the refusal is in the transcript, so the `talk` intent can answer it.

## Where to go next

- [`../03-llm-in-action/`](../03-llm-in-action/llm-in-action.ipynb) — the one-shot cycles this
  chat is built from, and a worked example of the failure that no validator catches.
- [`../01-create-a-btwin-graph/`](../01-create-a-btwin-graph/create-a-btwin-graph.ipynb) — the
  same kind of graph built entirely by hand.
- `Cycle.RDFChat` — the terminal chat. A few lines wire it to a graph on disk:

  ```python
  from btwin import RDF, Cycle

  BASE = "https://example.org/riverside/"
  graph = RDF.ByTTL("output/riverside.ttl", baseIRI=BASE)

  session = Cycle.RDFChat(graph, savePath="output/riverside_edited.ttl")

  print(f"{session['edits']} edit(s), written: {session['saved']}")
  ```